# Análise do Sistema de Descida Helike - Asas de TPU

Estudo paramétrico do sistema de autorrotação samara para descida controlada de PocketQube.

**Restrição:** Apenas asas de TPU (sem paraquedas complementares)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec
import warnings
warnings.filterwarnings('ignore')

# Configuração dos gráficos
plt.rcParams['figure.figsize'] = (12, 8)
plt.rcParams['font.size'] = 11
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3

## 1. Constantes e Modelo

In [ ]:
# Constantes físicas
g = 9.81          # m/s²
rho_ar = 1.225    # kg/m³

# Parâmetros do satélite
m_pq = 0.350      # kg (350g)
L_corpo = 0.05    # m (5cm)

# Parâmetros do material
rho_tpu = 1200    # kg/m³
espessura = 0.6e-3  # m (0.6mm)

def v_terminal(m, R, n, k=3.2):
    """Velocidade terminal de samara (m/s)"""
    v = k * np.sqrt(m) / (R * np.sqrt(n))
    return np.clip(v, 1.5, 20.0)

def massa_asas(R, n):
    """Massa total das asas (kg)"""
    area = 0.03 * R**2  # m² por asa
    return n * area * espessura * rho_tpu

def massa_total(R, n):
    """Massa total do sistema (kg)"""
    return m_pq + massa_asas(R, n)

def energia_impacto(m, v):
    """Energia cinética no impacto (J)"""
    return 0.5 * m * v**2

def velocidade_rotacao(R, v0, lambda_=0.065):
    """Velocidade angular (rad/s)"""
    omega = v0 / (lambda_ * R)
    return np.minimum(omega, 100)

print("Modelo carregado.")
print(f"Massa base do PocketQube: {m_pq*1000:.0f} g")
print(f"Material: TPU 95A (ρ={rho_tpu} kg/m³, esp={espessura*1000:.1f}mm)")

## 2. Velocidade Terminal vs Raio da Asa

Como o raio da asa afeta a velocidade de descida para diferentes números de asas.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

R_range = np.linspace(0.08, 0.30, 100)
n_asas_list = [1, 2, 3, 4, 6]
cores = ['#e74c3c', '#e67e22', '#f1c40f', '#2ecc71', '#3498db']

# Gráfico 1: Velocidade vs Raio
ax1 = axes[0]
for n, cor in zip(n_asas_list, cores):
    v_vals = [v_terminal(massa_total(R, n), R, n) for R in R_range]
    ax1.plot(R_range * 100, v_vals, linewidth=2, label=f'{n} asa(s)', color=cor)

ax1.axhline(y=5.5, color='red', linestyle='--', alpha=0.7, label='Limite seguro (5.5 m/s)')
ax1.axhline(y=8.0, color='orange', linestyle='--', alpha=0.7, label='Limite aceitável (8 m/s)')
ax1.axvspan(8, 16, alpha=0.1, color='green', label='Envelope dobrável (8-16cm)')
ax1.set_xlabel('Raio da Asa (cm)')
ax1.set_ylabel('Velocidade Terminal (m/s)')
ax1.set_title('Velocidade de Descida vs Raio da Asa')
ax1.legend(loc='upper right', fontsize=9)
ax1.set_ylim(0, 15)

# Gráfico 2: Energia vs Raio
ax2 = axes[1]
for n, cor in zip(n_asas_list, cores):
    m_vals = [massa_total(R, n) for R in R_range]
    v_vals = [v_terminal(m, R, n) for m, R in zip(m_vals, R_range)]
    E_vals = [energia_impacto(m, v) for m, v in zip(m_vals, v_vals)]
    ax2.plot(R_range * 100, E_vals, linewidth=2, label=f'{n} asa(s)', color=cor)

ax2.axhline(y=5, color='orange', linestyle='--', alpha=0.7, label='Limite energia (5 J)')
ax2.axvspan(8, 16, alpha=0.1, color='green')
ax2.set_xlabel('Raio da Asa (cm)')
ax2.set_ylabel('Energia de Impacto (J)')
ax2.set_title('Energia de Impacto vs Raio da Asa')
ax2.legend(loc='upper right', fontsize=9)
ax2.set_ylim(0, 25)

plt.tight_layout()
plt.show()

## 3. Mapa de Calor - Espaço de Design

Visualização 2D combinando raio e número de asas.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Malha de parâmetros
R_mesh = np.linspace(0.08, 0.25, 50)
n_mesh = np.array([1, 2, 3, 4, 6])
R_grid, N_grid = np.meshgrid(R_mesh, n_mesh)

# Calcular métricas
V_grid = np.zeros_like(R_grid)
E_grid = np.zeros_like(R_grid)
M_grid = np.zeros_like(R_grid)

for i in range(len(n_mesh)):
    for j in range(len(R_mesh)):
        R = R_mesh[j]
        n = n_mesh[i]
        m = massa_total(R, n)
        v = v_terminal(m, R, n)
        V_grid[i, j] = v
        E_grid[i, j] = energia_impacto(m, v)
        M_grid[i, j] = m * 1000

# Gráfico 1: Velocidade
ax1 = axes[0]
im1 = ax1.pcolormesh(R_grid * 100, N_grid, V_grid, cmap='RdYlGn_r', vmin=3, vmax=12)
ax1.contour(R_grid * 100, N_grid, V_grid, levels=[5, 6, 8], colors='black', linewidths=1)
ax1.set_xlabel('Raio (cm)')
ax1.set_ylabel('Número de Asas')
ax1.set_title('Velocidade Terminal (m/s)')
plt.colorbar(im1, ax=ax1, label='m/s')

# Gráfico 2: Energia
ax2 = axes[1]
im2 = ax2.pcolormesh(R_grid * 100, N_grid, E_grid, cmap='RdYlGn_r', vmin=1, vmax=15)
ax2.contour(R_grid * 100, N_grid, E_grid, levels=[3, 5, 10], colors='black', linewidths=1)
ax2.set_xlabel('Raio (cm)')
ax2.set_ylabel('Número de Asas')
ax2.set_title('Energia de Impacto (J)')
plt.colorbar(im2, ax=ax2, label='J')

# Gráfico 3: Massa
ax3 = axes[2]
im3 = ax3.pcolormesh(R_grid * 100, N_grid, M_grid, cmap='viridis', vmin=350, vmax=360)
ax3.set_xlabel('Raio (cm)')
ax3.set_ylabel('Número de Asas')
ax3.set_title('Massa Total (g)')
plt.colorbar(im3, ax=ax3, label='g')

plt.tight_layout()
plt.show()

## 4. Análise de Sensibilidade à Massa

Como a massa do satélite afeta o desempenho do sistema.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 10))

m_range = np.linspace(0.200, 0.500, 100)  # 200g a 500g
configs = [
    (0.12, 4, '4 asas 12cm'),
    (0.15, 4, '4 asas 15cm'),
    (0.16, 4, '4 asas 16cm'),
    (0.20, 4, '4 asas 20cm'),
]
cores = ['#e74c3c', '#e67e22', '#2ecc71', '#3498db']

# Gráfico 1: Velocidade vs Massa
ax1 = axes[0, 0]
for (R, n, label), cor in zip(configs, cores):
    v_vals = [v_terminal(m, R, n) for m in m_range]
    ax1.plot(m_range * 1000, v_vals, linewidth=2, label=label, color=cor)
ax1.axhline(y=5.5, color='red', linestyle='--', alpha=0.5)
ax1.axvline(x=350, color='gray', linestyle=':', alpha=0.5, label='Massa atual (350g)')
ax1.set_xlabel('Massa do Satélite (g)')
ax1.set_ylabel('Velocidade Terminal (m/s)')
ax1.set_title('Velocidade vs Massa')
ax1.legend(fontsize=9)

# Gráfico 2: Energia vs Massa
ax2 = axes[0, 1]
for (R, n, label), cor in zip(configs, cores):
    E_vals = [energia_impacto(m, v_terminal(m, R, n)) for m in m_range]
    ax2.plot(m_range * 1000, E_vals, linewidth=2, label=label, color=cor)
ax2.axhline(y=5, color='orange', linestyle='--', alpha=0.5)
ax2.axvline(x=350, color='gray', linestyle=':', alpha=0.5)
ax2.set_xlabel('Massa do Satélite (g)')
ax2.set_ylabel('Energia de Impacto (J)')
ax2.set_title('Energia vs Massa')
ax2.legend(fontsize=9)

# Gráfico 3: Velocidade de rotação vs Massa
ax3 = axes[1, 0]
for (R, n, label), cor in zip(configs, cores):
    omega_vals = [velocidade_rotacao(R, v_terminal(m, R, n)) for m in m_range]
    ax3.plot(m_range * 1000, omega_vals, linewidth=2, label=label, color=cor)
ax3.axvline(x=350, color='gray', linestyle=':', alpha=0.5)
ax3.set_xlabel('Massa do Satélite (g)')
ax3.set_ylabel('Velocidade Angular (rad/s)')
ax3.set_title('Rotação vs Massa')
ax3.legend(fontsize=9)

# Gráfico 4: Massa necessária para v0 = 5.5 m/s
ax4 = axes[1, 1]
R_range2 = np.linspace(0.10, 0.25, 50)
for n, cor in zip([2, 3, 4, 6], cores):
    # Para cada R, achar m tal que v_terminal(m, R, n) = 5.5
    # v = k * sqrt(m) / (R * sqrt(n))  =>  m = (v * R * sqrt(n) / k)²
    m_target = [(5.5 * R * np.sqrt(n) / 3.2)**2 * 1000 for R in R_range2]
    ax4.plot(R_range2 * 100, m_target, linewidth=2, label=f'{n} asa(s)', color=cor)
ax4.axhline(y=350, color='gray', linestyle=':', alpha=0.5, label='Massa atual')
ax4.axhline(y=300, color='green', linestyle='--', alpha=0.5, label='Meta 300g')
ax4.set_xlabel('Raio da Asa (cm)')
ax4.set_ylabel('Massa Máxima (g)')
ax4.set_title('Massa Máxima para v₀ = 5.5 m/s')
ax4.legend(fontsize=9)

plt.tight_layout()
plt.show()

## 5. Perfil da Asa Samara

Visualização da geometria recomendada.

In [ ]:
def perfil_samara(r, R):
    """Corda da asa samara na posição r"""
    if r <= 0 or r >= R:
        return 0
    x = r / R
    # Forma: máximo em ~30% do raio
    return 0.08 * R * 4 * x * (1 - x)**2

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# Gráfico 1: Perfis para diferentes raios
ax1 = axes[0]
for R_cm in [12, 15, 16, 20]:
    R = R_cm / 100
    r = np.linspace(0, R, 100)
    w = [perfil_samara(ri, R) for ri in r]
    ax1.plot(r * 100, [wi * 1000 for wi in w], linewidth=2, label=f'R={R_cm}cm')
ax1.set_xlabel('Posição radial (cm)')
ax1.set_ylabel('Corda (mm)')
ax1.set_title('Perfil da Asa Samara')
ax1.legend()

# Gráfico 2: Visualização 2D da asa recomendada
ax2 = axes[1]
R_rec = 0.16  # 16cm
theta = np.linspace(0, 2*np.pi, 400)
for offset in [0, np.pi/2, np.pi, 3*np.pi/2]:  # 4 asas
    r_plot = np.linspace(0, R_rec, 100)
    w_plot = np.array([perfil_samara(r, R_rec) for r in r_plot])
    # Coordenadas da asa
    x_asa = r_plot * np.cos(offset)
    y_asa = r_plot * np.sin(offset)
    # Largura perpendicular
    x_l = x_asa - w_plot * np.sin(offset) / 2
    x_r = x_asa + w_plot * np.sin(offset) / 2
    y_l = y_asa + w_plot * np.cos(offset) / 2
    y_r = y_asa - w_plot * np.cos(offset) / 2
    ax2.fill_betweenx(y_asa, x_l, x_r, alpha=0.5, color='#2ecc71')
    ax2.plot(x_l, y_l, 'g-', linewidth=1)
    ax2.plot(x_r, y_r, 'g-', linewidth=1)

# Corpo central
corpo = plt.Rectangle((-0.025, -0.025), 0.05, 0.05, 
                       fill=True, facecolor='#34495e', edgecolor='black')
ax2.add_patch(corpo)
ax2.set_xlim(-0.20, 0.20)
ax2.set_ylim(-0.20, 0.20)
ax2.set_aspect('equal')
ax2.set_xlabel('x (m)')
ax2.set_ylabel('y (m)')
ax2.set_title('Configuração: 4 Asas de 16cm')

# Gráfico 3: Área vs Raio
ax3 = axes[2]
R_range3 = np.linspace(0.08, 0.25, 50)
for n in [1, 2, 4]:
    areas = []
    for R in R_range3:
        r_int = np.linspace(1e-6, R-1e-6, 100)
        w_int = [perfil_samara(r, R) for r in r_int]
        area = np.trapz(w_int, r_int) * n * 10000  # cm²
        areas.append(area)
    ax3.plot(R_range3 * 100, areas, linewidth=2, label=f'{n} asa(s)')
ax3.set_xlabel('Raio (cm)')
ax3.set_ylabel('Área Total (cm²)')
ax3.set_title('Área Aerodinâmica vs Raio')
ax3.legend()

plt.tight_layout()
plt.show()

## 6. Análise de Tempo de Descida

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Gráfico 1: Tempo vs Altura para diferentes configurações
ax1 = axes[0]
alturas = np.linspace(100, 5000, 100)  # 100m a 5000m
configs_tempo = [
    (0.12, 4, '4 asas 12cm'),
    (0.15, 4, '4 asas 15cm'),
    (0.16, 4, '4 asas 16cm'),
    (0.20, 4, '4 asas 20cm'),
    (0.15, 6, '6 asas 15cm'),
]

for R, n, label in configs_tempo:
    m = massa_total(R, n)
    v0 = v_terminal(m, R, n)
    tempos = alturas / v0 / 60  # minutos
    ax1.plot(alturas, tempos, linewidth=2, label=f'{label} (v₀={v0:.1f} m/s)')

ax1.axvline(x=1000, color='gray', linestyle=':', alpha=0.5, label='Referência 1000m')
ax1.set_xlabel('Altura de Liberação (m)')
ax1.set_ylabel('Tempo de Descida (min)')
ax1.set_title('Tempo de Descida vs Altura')
ax1.legend(fontsize=9)
ax1.set_ylim(0, 15)

# Gráfico 2: Velocidade vs Tempo (aceleração)
ax2 = axes[1]
t = np.linspace(0, 30, 200)  # 0 a 30 segundos

for R, n, label in configs_tempo[:3]:
    m = massa_total(R, n)
    v_term = v_terminal(m, R, n)
    tau = 2.0  # constante de tempo (s)
    v = v_term * (1 - np.exp(-t / tau))
    ax2.plot(t, v, linewidth=2, label=label)

ax2.axhline(y=5.5, color='red', linestyle='--', alpha=0.5, label='v₀ alvo')
ax2.set_xlabel('Tempo após liberação (s)')
ax2.set_ylabel('Velocidade (m/s)')
ax2.set_title('Aceleração até Velocidade Terminal')
ax2.legend(fontsize=9)

plt.tight_layout()
plt.show()

## 7. Comparação de Todas as Configurações Viáveis

In [ ]:
# Gerar todas as combinações
raios = [0.10, 0.12, 0.14, 0.15, 0.16, 0.18, 0.20, 0.22, 0.25]
num_asas = [1, 2, 3, 4, 6]

resultados = []
for n in num_asas:
    for R in raios:
        m = massa_total(R, n)
        v0 = v_terminal(m, R, n)
        E = energia_impacto(m, v0)
        omega = velocidade_rotacao(R, v0)
        
        # Verificar se cabe dobrada (heurística)
        espaco = R / n + 0.005
        cabe = espaco <= 0.05
        
        resultados.append({
            'n': n, 'R': R, 'm': m, 'v0': v0, 'E': E,
            'omega': omega, 'cabe': cabe
        })

# Filtrar apenas as que cabem
configs_viaveis = [r for r in resultados if r['cabe']]

# Gráfico de barras
fig, axes = plt.subplots(2, 1, figsize=(14, 10))

# Preparar dados
labels = [f"{r['n']}×R{r['R']*100:.0f}" for r in configs_viaveis]
v0_vals = [r['v0'] for r in configs_viaveis]
E_vals = [r['E'] for r in configs_viaveis]

# Cores baseadas na classificação
cores_v0 = ['#2ecc71' if v < 5.5 else '#f1c40f' if v < 7 else '#e74c3c' for v in v0_vals]
cores_E = ['#2ecc71' if e < 3 else '#f1c40f' if e < 6 else '#e74c3c' for e in E_vals]

# Gráfico 1: Velocidade
ax1 = axes[0]
bars1 = ax1.bar(labels, v0_vals, color=cores_v0, edgecolor='black', linewidth=0.5)
ax1.axhline(y=5.5, color='red', linestyle='--', linewidth=2, label='Limite seguro')
ax1.set_ylabel('Velocidade Terminal (m/s)')
ax1.set_title('Velocidade de Descida por Configuração (apenas configs que cabem)')
ax1.legend()
plt.setp(ax1.xaxis.get_majorticklabels(), rotation=45, ha='right')

# Adicionar valores nas barras
for bar, val in zip(bars1, v0_vals):
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.1,
             f'{val:.1f}', ha='center', fontsize=8)

# Gráfico 2: Energia
ax2 = axes[1]
bars2 = ax2.bar(labels, E_vals, color=cores_E, edgecolor='black', linewidth=0.5)
ax2.axhline(y=5, color='orange', linestyle='--', linewidth=2, label='Limite energia')
ax2.set_ylabel('Energia de Impacto (J)')
ax2.set_title('Energia de Impacto por Configuração')
ax2.legend()
plt.setp(ax2.xaxis.get_majorticklabels(), rotation=45, ha='right')

for bar, val in zip(bars2, E_vals):
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.1,
             f'{val:.1f}', ha='center', fontsize=8)

plt.tight_layout()
plt.show()

## 8. Resumo e Recomendação

In [ ]:
# Encontrar melhor configuração que cabe
configs_viaveis.sort(key=lambda x: x['v0'])
melhor = configs_viaveis[0]

print("=" * 70)
print("RECOMENDAÇÃO FINAL")
print("=" * 70)
print(f""
Configuração ótima (que cabe no envelope 5×5×5cm):

  Número de asas:      {melhor['n']}
  Raio de cada asa:    {melhor['R']*100:.0f} cm
  Massa total:         {melhor['m']*1000:.1f} g
  Velocidade descida:  {melhor['v0']:.2f} m/s ({melhor['v0']*3.6:.1f} km/h)
  Energia impacto:     {melhor['E']:.2f} J
  Rotação estimada:    {melhor['omega']:.1f} rad/s ({melhor['omega']/(2*np.pi):.1f} Hz)

Classificação: {'ACEITÁVEL' if melhor['v0'] < 6 else 'MODERADO'}
""")

print("\nTop 5 configurações viáveis:")
print(f"{'Config':<15} | {'v₀ (m/s)':>8} | {'E (J)':>6} | {'ω (Hz)':>7}")
print("-" * 45)
for r in configs_viaveis[:5]:
    print(f"{r['n']}×R{r['R']*100:.0f}cm{'':<8} | {r['v0']:>8.2f} | {r['E']:>6.1f} | {r['omega']/(2*np.pi):>6.1f}")

print("\n" + "=" * 70)